In [2]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

### Dataset

In [3]:
from datasets import load_dataset
dataset_name = "ridwanFatur98/synth-invoice"
train_dataset = load_dataset(dataset_name, split="train")
sample = train_dataset[0]

### Get Encoder Outputs

In [4]:
from timm.models.swin_transformer import SwinTransformer
import torch.nn as nn
from typing import List

from scripts.image_process import process_image, image_to_tensor
from scripts.swin_encoder import SwinEncoder

# Input Config
input_size = [1280, 960]
align_long_axis = False
random_padding = True

# Input Encoder
img = process_image(sample["image"], input_size, align_long_axis, random_padding)
image_tensors = image_to_tensor(img)
image_tensors = image_tensors.unsqueeze(0)
image_tensors = image_tensors.to(device)

# Encoder Config
encoder_layer = [1, 1, 1, 1]
window_size = 10
input_size = [1280, 960]

# Encoder
encoder = SwinEncoder(
    input_size=input_size,
    window_size=window_size,
    encoder_layer=encoder_layer,
).to(device)

# Inference
encoder_outputs = encoder(image_tensors)

In [5]:
image_tensors.shape

torch.Size([1, 3, 1280, 960])

In [6]:
encoder_outputs.shape

torch.Size([1, 40, 30, 1024])

### Tokenizer

In [7]:
from scripts.tokenizer import get_tokenizer

In [8]:
tokenizer = get_tokenizer()

In [9]:
len(tokenizer)

57568

### BART Decoder Model

In [11]:
from typing import Any, List, Optional, Union
from transformers import MBartConfig, MBartForCausalLM
from transformers.file_utils import ModelOutput
from scripts.bart_decoder import BARTDecoder

In [12]:
max_position_embeddings = 768
decoder_layer = 1

In [13]:
decoder = BARTDecoder(
    max_position_embeddings=max_position_embeddings,
    decoder_layer=decoder_layer,
    tokenizer=tokenizer
).to(device)

In [14]:
sum(p.numel() for p in decoder.parameters())

76538880

### Prepare Input For Decoder

In [22]:
from scripts.ground_truth_process import process_ground_truth, get_train_inputs

task_start_token = '<s_cord-v2>'
prompt_end_token= '<s_cord-v2>'

max_length = 768
ignore_id: int = -100
prompt_end_token_id = tokenizer.convert_tokens_to_ids(prompt_end_token)

ground_truth = sample["ground_truth"]
gt_token_sequence, _ = process_ground_truth(
    tokenizer,
    ground_truth,
    task_start_token=task_start_token,
)
processed_parse = gt_token_sequence 

input_ids, labels = get_train_inputs(tokenizer, processed_parse, prompt_end_token_id, max_length, ignore_id)
input_ids = input_ids.unsqueeze(0).to(device)
labels = labels.unsqueeze(0).to(device)
labels.shape, input_ids.shape, encoder_outputs.shape

(torch.Size([1, 768]), torch.Size([1, 768]), torch.Size([1, 40, 30, 1024]))

In [23]:
decoder_input_ids = input_ids[:, :-1]
decoder_labels = labels[:, 1:]

decoder_input_ids.shape, decoder_labels.shape

(torch.Size([1, 767]), torch.Size([1, 767]))

In [24]:
decoder_outputs = decoder(
    input_ids=decoder_input_ids,
    encoder_hidden_states=encoder_outputs.flatten(1, 2),
    labels=decoder_labels,
)

In [25]:
decoder_outputs['loss']

tensor(16.4999, device='cuda:0', grad_fn=<NllLossBackward0>)